# Pipeline v20.1 -- PURE QWEN inference/eval
**EXACT 2026 -- data v5 strict-clean**

Final answer path is **pure Qwen3-8B LoRA v20** only:
- no Z3
- no hybrid
- no anti-overclaim
- no LGBM
- BoN self-consistency: N=5, temp=0.5, top_p=0.95


In [1]:
# ==================================================================
# STAGE 0 -- vLLM compatibility installer (Kaggle T4)
# PATCH: disable FlashInfer BEFORE any vLLM import.
# FlashInfer JIT often fails on Kaggle Tesla T4 (sm_75).
# ==================================================================
import os, sys, subprocess, importlib, shutil, glob

# Must be set before importing vLLM
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Prefer Triton attention and physically hide FlashInfer so vLLM cannot pick it.
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["VLLM_USE_V1"] = "0"  # harmless if ignored by newer vLLM

def _disable_flashinfer():
    shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
    shutil.rmtree("/root/.cache/torch_extensions", ignore_errors=True)
    for d in glob.glob("/usr/local/lib/python3.12/dist-packages/flashinfer*"):
        tgt = d + "_DISABLED"
        if os.path.exists(d) and not os.path.exists(tgt):
            try:
                os.rename(d, tgt)
                print("Disabled FlashInfer:", d, flush=True)
            except Exception as e:
                print("Could not disable FlashInfer:", d, repr(e), flush=True)

_disable_flashinfer()

def _pip(*args):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"] + list(args),
        capture_output=True, text=True
    )

def _clear_vllm():
    for mod in list(sys.modules.keys()):
        if any(x in mod for x in ("vllm", "transformers", "flashinfer", "torch._dynamo", "torch._inductor")):
            del sys.modules[mod]

def _try_import():
    _disable_flashinfer()
    try:
        from vllm import LLM, SamplingParams
        return True, ""
    except Exception as e:
        return False, str(e).split("\n")[0][:180]

print("="*80)
print("STAGE 0 vLLM compatibility installer -- FlashInfer disabled")
print("="*80)

ok, msg = _try_import()
if ok:
    print("Pre-installed vLLM: OK", flush=True)
    _vllm_ok = True
else:
    print(f"Pre-installed vLLM: FAIL ({msg})", flush=True)
    # Prefer versions previously stable with T4 + no FlashInfer. Try latest only after disabling FlashInfer.
    PAIRS = [
        ("4.48.0", ""),             # latest vLLM, FlashInfer hidden
        ("4.57.6", "0.22.1"),
        ("4.46.3", "0.9.1"),
        ("4.46.3", "0.8.5.post1"),
        ("4.44.2", "0.8.5.post1"),
        ("4.44.2", "0.7.3"),
        ("4.46.3", "0.6.6.post1"),
    ]
    _vllm_ok = False
    for tv, vv in PAIRS:
        vllm_pkg = f"vllm=={vv}" if vv else "vllm"
        print(f"  Trying transformers=={tv} + {vllm_pkg}...", end=" ", flush=True)
        _pip("protobuf==5.29.5")
        _pip(f"transformers=={tv}")
        _pip(vllm_pkg)

        # If pip reinstalled FlashInfer as a dependency, hide it again before import.
        _disable_flashinfer()
        _clear_vllm()
        ok, msg = _try_import()
        if ok:
            print("OK", flush=True)
            _vllm_ok = True
            break
        else:
            print(f"FAIL ({msg})", flush=True)
    if not _vllm_ok:
        raise RuntimeError("No vLLM+transformers pair worked. Restart session and retry, or use transformers fallback.")

# Final imports
import json, re, csv, time, random, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer
import vllm as _vllm_mod
import transformers as _tfm

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("\n" + "="*80)
print("IMPORT SUMMARY")
print("="*80)
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA OK      : {torch.cuda.is_available()}")
print(f"N_GPUS       : {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}        : {props.name} ({props.total_memory/1024**3:.1f} GB)")
print(f"vLLM version : {_vllm_mod.__version__}")
print(f"Transformers : {_tfm.__version__}")
print("Attention backend requested: TRITON_ATTN; FlashInfer package hidden")
print("All imports OK", flush=True)


STAGE 0 vLLM compatibility installer -- FlashInfer disabled
Pre-installed vLLM: FAIL (No module named 'vllm')
  Trying transformers==4.48.0 + vllm... Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_python-0.6.11.post2.dist-info
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_cubin-0.6.11.post2.dist-info
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_cubin
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_DISABLED
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_python-0.6.11.post2.dist-info_DISABLED
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_cubin_DISABLED
Disabled FlashInfer: /usr/local/lib/python3.12/dist-packages/flashinfer_cubin-0.6.11.post2.dist-info_DISABLED
WARNING 06-07 08:11:29 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepat

In [2]:

# ==================================================================
# BOOT CHECK
# ==================================================================
import os, sys, glob, torch
print("="*70)
print("BOOT CHECK")
print("="*70)
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "CUDA available=", torch.cuda.is_available())
print("N_GPUS:", N_GPUS)
print("/kaggle/input entries:")
for p in sorted(glob.glob("/kaggle/input/*"))[:30]:
    print(" -", p)
print("="*70)


BOOT CHECK
Python: 3.12.13
Torch : 2.11.0+cu130 CUDA available= True
N_GPUS: 2
/kaggle/input entries:
 - /kaggle/input/datasets
 - /kaggle/input/models
 - /kaggle/input/notebooks


In [3]:

# ==================================================================
# LOCATE + UNZIP LoRA v20
# ==================================================================
import os, glob, zipfile

LORA_ZIP_GLOB = "/kaggle/input/**/qwen3_cot_lora_v20_v5.zip"
UNZIP_DEST = "/kaggle/working"
zip_hits = sorted(glob.glob(LORA_ZIP_GLOB, recursive=True))

print("="*70)
print("LOCATE LoRA v20")
print("="*70)
print("zip hits:", zip_hits[:5])

resolved = None
if zip_hits:
    zpath = zip_hits[0]
    print("Found zip:", zpath)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(UNZIP_DEST)
    cand = os.path.join(UNZIP_DEST, "qwen3_cot_lora_v20_v5")
    if os.path.exists(os.path.join(cand, "adapter_config.json")):
        resolved = cand
    else:
        for root, _, files in os.walk(UNZIP_DEST):
            if "adapter_config.json" in files and "qwen3_cot_lora_v20" in root:
                resolved = root
                break
else:
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files and "qwen3_cot_lora_v20" in root:
            resolved = root
            break

assert resolved is not None, "Cannot find qwen3_cot_lora_v20_v5.zip or unzipped adapter. Add v20 finetune output as input."
assert os.path.exists(os.path.join(resolved, "adapter_config.json")), resolved
assert os.path.exists(os.path.join(resolved, "adapter_model.safetensors")), resolved

COT_LORA_PATH = resolved
print("[OK] COT_LORA_PATH =", COT_LORA_PATH)
print("[OK] adapter_model.safetensors MB =", os.path.getsize(os.path.join(resolved, "adapter_model.safetensors"))/1e6)


LOCATE LoRA v20
zip hits: ['/kaggle/input/notebooks/nguyenminhtric/v20-finetune/qwen3_cot_lora_v20_v5.zip']
Found zip: /kaggle/input/notebooks/nguyenminhtric/v20-finetune/qwen3_cot_lora_v20_v5.zip
[OK] COT_LORA_PATH = /kaggle/working/qwen3_cot_lora_v20_v5
[OK] adapter_model.safetensors MB = 174.655536


In [4]:

# ==================================================================
# CONFIG + PATH RESOLVER
# ==================================================================
import os, glob, random, json, re, csv, time
from collections import Counter, defaultdict

def _resolve(cands, label="path"):
    expanded = []
    for p in cands:
        hits = sorted(glob.glob(p, recursive=True)) if any(ch in p for ch in "*?[") else [p]
        expanded.extend(hits)
    for p in expanded:
        if os.path.exists(p):
            print(f"resolved {label}: {p}")
            return p
    print(f"WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

DATASET_PATH = _resolve([
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "DATASET")

TEST_PATH = _resolve([
    "/kaggle/input/**/generated_v4style_300.json",
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")

QWEN_MODEL_ID = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"

SEED = 42
TRAIN_RATIO = 0.80
QWEN_BON_N = 5
QWEN_BON_TEMP = 0.5
QWEN_TOP_P = 0.95
COT_MAX_TOKENS = 768
TENSOR_PARALLEL = min(max(N_GPUS, 1), 2)
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTIL = 0.85
OUTPUT_JSON = "/kaggle/working/pipeline_results_20_1_pureqwen_v5.json"
SUBMISSION_CSV = "/kaggle/working/submission_v20_pureqwen.csv"
SUBMISSION_JSON = "/kaggle/working/submission_v20_pureqwen.json"

V20_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

print("="*70)
print("PURE QWEN CONFIG")
print("="*70)
print("DATASET_PATH:", DATASET_PATH)
print("TEST_PATH   :", TEST_PATH)
print("COT_LORA_PATH:", COT_LORA_PATH)
print("QWEN_BON_N:", QWEN_BON_N, "temp:", QWEN_BON_TEMP, "top_p:", QWEN_TOP_P)
print("TENSOR_PARALLEL:", TENSOR_PARALLEL)
print("NO Z3 / NO HYBRID / NO LGBM")


resolved DATASET: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
resolved TEST: /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
PURE QWEN CONFIG
DATASET_PATH: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
TEST_PATH   : /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
COT_LORA_PATH: /kaggle/working/qwen3_cot_lora_v20_v5
QWEN_BON_N: 5 temp: 0.5 top_p: 0.95
TENSOR_PARALLEL: 2
NO Z3 / NO HYBRID / NO LGBM


In [5]:
# ==================================================================
# STAGE 0B -- Confirm FlashInfer is disabled
# Do NOT force FLASHINFER here. This notebook uses Triton attention on T4.
# ==================================================================
import os, shutil, glob

os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
shutil.rmtree("/root/.cache/torch_extensions", ignore_errors=True)

print("FlashInfer remains disabled; using TRITON_ATTN for Kaggle T4.")
print("If vLLM was imported before this cell in the same kernel, restart and Run All.")


FlashInfer remains disabled; using TRITON_ATTN for Kaggle T4.
If vLLM was imported before this cell in the same kernel, restart and Run All.


In [6]:
# ==================================================================
# LOAD vLLM + TOKENIZER
# ==================================================================
import os, time, torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

print("="*70)
print("Loading vLLM engine...")
print("="*70)

print("torch cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

try:
    N_GPUS
except NameError:
    N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

TENSOR_PARALLEL = min(max(N_GPUS, 1), 2)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)

llm = LLM(
    model=QWEN_MODEL_ID,
    tokenizer=QWEN_MODEL_ID,
    trust_remote_code=True,
    dtype="float16",
    tensor_parallel_size=TENSOR_PARALLEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    enable_lora=True,
    max_lora_rank=16,
    enforce_eager=True,
    disable_log_stats=True,
)

LORA_REQ = LoRARequest("v20_cot_lora", 1, COT_LORA_PATH)

print(f"vLLM loaded in {(time.time()-t0):.1f}s")
print("LoRA enabled:", COT_LORA_PATH)

Loading vLLM engine...
torch cuda: True
device_count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
INFO 06-07 08:11:37 [utils.py:278] non-default args: {'tokenizer': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'}
WARNING 06-07 08:11:37 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND
WARNING 06-07 08:11:37 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-07 08:11:53 [model.py:617] Resolved architecture: Qwen3ForCausalLM
WARNING 06-07 08:11:53 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 06-07 08:11:53 [model.py:1752] Using max model len 8192
INFO 06-07 08:11:53 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(Worker_TP0 pid=266) INFO 06-07 08:12:33 [weight_utils.py:856] Prefetching checkpoint files: 10% (1/3)


Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:19<01:17, 19.26s/it]


(Worker_TP0 pid=266) INFO 06-07 08:12:43 [weight_utils.py:856] Prefetching checkpoint files: 20% (2/3)
(Worker_TP1 pid=267) INFO 06-07 08:12:45 [weight_utils.py:856] Prefetching checkpoint files: 10% (1/2)


Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:24<00:33, 11.09s/it]


(Worker_TP1 pid=267) INFO 06-07 08:12:49 [weight_utils.py:856] Prefetching checkpoint files: 20% (2/2)
(Worker_TP1 pid=267) INFO 06-07 08:12:49 [weight_utils.py:879] Prefetching checkpoint files into page cache finished in 24.64s
(Worker_TP0 pid=266) INFO 06-07 08:12:50 [weight_utils.py:856] Prefetching checkpoint files: 30% (3/3)
(Worker_TP0 pid=266) INFO 06-07 08:12:50 [weight_utils.py:879] Prefetching checkpoint files into page cache finished in 25.67s


Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:27<00:14,  7.44s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:29<00:05,  5.38s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:31<00:00,  3.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:31<00:00,  6.22s/it]
(Worker_TP0 pid=266) 


(Worker_TP0 pid=266) INFO 06-07 08:12:55 [default_loader.py:397] Loading weights took 31.15 seconds
(Worker_TP0 pid=266) INFO 06-07 08:12:55 [punica_selector.py:20] Using PunicaWrapperGPU.
(Worker_TP1 pid=267) INFO 06-07 08:12:56 [model_runner.py:295] Model loading took 7.71 GiB and 32.691357 seconds
(Worker_TP0 pid=266) INFO 06-07 08:12:56 [model_runner.py:295] Model loading took 7.71 GiB and 32.745203 seconds
(Worker_TP0 pid=266) WARNING 06-07 08:13:02 [utils.py:279] Using default LoRA kernel configs
(Worker_TP0 pid=266) INFO 06-07 08:13:16 [gpu_worker.py:466] Available KV cache memory: 3.92 GiB
(EngineCore pid=240) INFO 06-07 08:13:16 [kv_cache_utils.py:1733] GPU KV cache size: 57,120 tokens
(EngineCore pid=240) INFO 06-07 08:13:16 [kv_cache_utils.py:1734] Maximum concurrency for 8,192 tokens per request: 6.97x
(Worker_TP0 pid=266) INFO 06-07 08:13:37 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(Worker_TP1 p

In [7]:

# ==================================================================
# DATA LOADING + PROMPT BUILDING
# ==================================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

raw = load_json(DATASET_PATH)
print(f"Loaded labeled data: {len(raw)} records")

rng = random.Random(SEED)
ids = list(range(len(raw)))
rng.shuffle(ids)
n_train = int(len(raw) * TRAIN_RATIO)
train_ids = set(ids[:n_train])
val_ids = set(ids[n_train:])

def iter_questions(samples, split_name, keep_gold=True):
    rows = []
    for sid, s in enumerate(samples):
        nls = s.get("premises-NL", [])
        qs = s.get("questions", [])
        ans = s.get("answers", [])
        exps = s.get("explanation", [])
        for q_idx, q in enumerate(qs):
            gold = str(ans[q_idx]).strip() if keep_gold and q_idx < len(ans) else None
            exp  = str(exps[q_idx]) if q_idx < len(exps) else ""
            rows.append({
                "sample_id": sid, "q_idx": q_idx, "split": split_name,
                "premises_nl": nls, "question": q, "gold": gold, "explanation": exp
            })
    return rows

train_rows = iter_questions([raw[i] for i in range(len(raw)) if i in train_ids], "train", True)
val_rows   = iter_questions([raw[i] for i in range(len(raw)) if i in val_ids], "val", True)
all_rows   = iter_questions(raw, "all", True)

print(f"Train questions: {len(train_rows)}")
print(f"Val questions  : {len(val_rows)}")
print(f"All questions  : {len(all_rows)}")
print("Val label distribution:", Counter(r["gold"] for r in val_rows))

def build_user_msg(premises_nl, question):
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

def to_chat_prompt(row):
    messages = [
        {"role": "system", "content": V20_COT_SYSTEM},
        {"role": "user", "content": build_user_msg(row["premises_nl"], row["question"])},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


Loaded labeled data: 399 records
Train questions: 628
Val questions  : 156
All questions  : 784
Val label distribution: Counter({'No': 40, 'Unknown': 35, 'Yes': 34, 'A': 28, 'B': 8, 'C': 6, 'D': 5})


In [8]:

# ==================================================================
# PURE QWEN BoN GENERATION + ANSWER EXTRACTION
# ==================================================================
VALID = {"YES":"Yes", "NO":"No", "UNKNOWN":"Unknown", "A":"A", "B":"B", "C":"C", "D":"D"}

def normalize_answer(x):
    if x is None:
        return "Unknown"
    s = str(x).strip()
    u = s.upper()
    if u in VALID:
        return VALID[u]
    if u in ("TRUE", "T"):
        return "Yes"
    if u in ("FALSE", "F"):
        return "No"
    if "UNKNOWN" in u or "CANNOT BE DETERMINED" in u or "NOT ENOUGH" in u:
        return "Unknown"
    return "Unknown"

def extract_final_answer(text):
    if not text:
        return "Unknown"
    # Prefer explicit final answer marker.
    pats = [
        r"Final\s*Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b",
        r"Final\s*answer\s*is\s*(Yes|No|Unknown|A|B|C|D)\b",
        r"Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b",
    ]
    for pat in pats:
        m = re.search(pat, text, flags=re.I)
        if m:
            return normalize_answer(m.group(1))
    # Fallback: last occurrence of a valid standalone label near the end.
    tail = text[-300:]
    hits = re.findall(r"\b(Yes|No|Unknown|A|B|C|D)\b", tail, flags=re.I)
    return normalize_answer(hits[-1]) if hits else "Unknown"

def majority_vote(ans_list):
    norm = [normalize_answer(a) for a in ans_list]
    cnt = Counter(norm)
    # deterministic tie-break: first answer among tied labels
    best_n = max(cnt.values()) if cnt else 0
    tied = {k for k,v in cnt.items() if v == best_n}
    for a in norm:
        if a in tied:
            return a, cnt
    return "Unknown", cnt

def batch_predict(rows, batch_size=128):
    prompts = [to_chat_prompt(r) for r in rows]
    params = SamplingParams(
        n=QWEN_BON_N,
        temperature=QWEN_BON_TEMP,
        top_p=QWEN_TOP_P,
        max_tokens=COT_MAX_TOKENS,
        repetition_penalty=1.05,
    )
    results = []
    t0 = time.time()
    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        outs = llm.generate(prompts[start:end], params, lora_request=LORA_REQ)
        for row, out in zip(rows[start:end], outs):
            texts = [o.text for o in out.outputs]
            ans_list = [extract_final_answer(t) for t in texts]
            pred, votes = majority_vote(ans_list)
            rec = dict(row)
            rec.update({
                "qwen_answer": pred,
                "qwen_votes": dict(votes),
                "qwen_sc_conf": max(votes.values()) / max(1, sum(votes.values())),
                "qwen_raw_0": texts[0] if texts else "",
            })
            results.append(rec)
        print(f"[{end}/{len(prompts)}] elapsed={(time.time()-t0)/60:.1f} min", flush=True)
    return results


In [9]:

# ==================================================================
# RUN EVAL ON TRAIN / VAL / FULL
# ==================================================================
train_pred = batch_predict(train_rows)
val_pred   = batch_predict(val_rows)
# Avoid re-running full separately: combine train+val predictions.
all_pred = train_pred + val_pred

def _U(x): return str(x).strip().upper()

def compute_metrics(rows):
    preds = [_U(r["qwen_answer"]) for r in rows]
    golds = [_U(r["gold"]) for r in rows]
    labels = sorted(set(golds))
    per = {}
    for lab in labels:
        tp = sum(1 for p,g in zip(preds,golds) if p==lab and g==lab)
        fp = sum(1 for p,g in zip(preds,golds) if p==lab and g!=lab)
        fn = sum(1 for p,g in zip(preds,golds) if p!=lab and g==lab)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab] = {"precision":prec, "recall":rec, "f1":f1, "n":sum(1 for g in golds if g==lab)}
    acc = sum(1 for p,g in zip(preds,golds) if p==g) / len(rows) if rows else 0.0
    macro_f1 = sum(v["f1"] for v in per.values()) / len(per) if per else 0.0
    weighted_f1 = sum(v["f1"]*v["n"] for v in per.values()) / len(rows) if rows else 0.0
    mc_labels = {"A","B","C","D"}
    mc_rows = [r for r in rows if _U(r["gold"]) in mc_labels]
    ynu_rows = [r for r in rows if _U(r["gold"]) in {"YES","NO","UNKNOWN"}]
    return {
        "acc": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "per_class": per,
        "mc_acc": (sum(_U(r["qwen_answer"])==_U(r["gold"]) for r in mc_rows)/len(mc_rows) if mc_rows else 0.0),
        "ynu_acc": (sum(_U(r["qwen_answer"])==_U(r["gold"]) for r in ynu_rows)/len(ynu_rows) if ynu_rows else 0.0),
        "n": len(rows),
    }

m_train = compute_metrics(train_pred)
m_val   = compute_metrics(val_pred)
m_full  = compute_metrics(all_pred)

def f1(m, lab):
    return m["per_class"].get(lab, {}).get("f1", 0.0)

print("\n" + "="*100)
print("v20.1 PURE QWEN -- metrics")
print("="*100)
print(f"{'Split':8s} | {'acc':>7s} {'mF1':>7s} {'wF1':>7s} {'Yes-F1':>7s} {'No-F1':>7s} {'Unk-F1':>7s} {'MC-acc':>7s}")
print("-"*100)
for name,m in [("TRAIN",m_train),("VAL",m_val),("FULL",m_full)]:
    print(f"{name:8s} | {m['acc']*100:6.1f}% {m['macro_f1']:7.3f} {m['weighted_f1']:7.3f} "
          f"{f1(m,'YES'):7.3f} {f1(m,'NO'):7.3f} {f1(m,'UNKNOWN'):7.3f} {m['mc_acc']*100:6.1f}%")
print("="*100)

print("\nPer-class VAL:")
for lab, d in m_val["per_class"].items():
    print(f"  {lab:>8s}: f1={d['f1']:.3f}  prec={d['precision']:.3f}  rec={d['recall']:.3f}  n={d['n']}")


Rendering prompts:   0%|          | 0/128 [00:00<?, ?it/s]

WARNING 06-07 08:13:40 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/640 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=266) WARNING 06-07 08:13:43 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _lora_expand_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=266) WARNING 06-07 08:13:43 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _lora_shrink_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=266) WARNING 06-07 08:13:45 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=266) WARNING 06-07 08:13:56 [jit_monitor.py:103] Triton kernel JIT compilation during inference: reduce_segments. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=266) WARNING 06-07 08:14:04 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel.

Processed prompts: 100%|██████████| 640/640 [04:24<00:00,  2.42it/s, est. speed input: 807.01 toks/s, output: 163.63 toks/s] 

...

Processed prompts: 100%|██████████| 640/640 [02:21<00:00,  4.53it/s, est. speed input: 1508.30 toks/s, output: 286.39 toks/s]

[128/156] elapsed=2.4 min


Rendering prompts:   0%|          | 0/28 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 140/140 [00:50<00:00,  2.78it/s, est. speed input: 877.06 toks/s, output: 170.67 toks/s] 

[156/156] elapsed=3.2 min

v20.1 PURE QWEN -- metrics
Split    |     acc     mF1     wF1  Yes-F1   No-F1  Unk-F1  MC-acc
----------------------------------------------------------------------------------------------------
TRAIN    |   57.5%   0.554   0.561   0.585   0.656   0.388   78.2%
VAL      |   61.5%   0.641   0.597   0.620   0.625   0.408   83.0%
FULL     |   58.3%   0.574   0.568   0.591   0.650   0.392   79.4%

Per-class VAL:
         A: f1=0.687  prec=0.590  rec=0.821  n=28
         B: f1=0.778  prec=0.700  rec=0.875  n=8
         C: f1=0.533  prec=0.444  rec=0.667  n=6
         D: f1=0.833  prec=0.714  rec=1.000  n=5
        NO: f1=0.625  prec=0.625  rec=0.625  n=40
   UNKNOWN: f1=0.408  prec=0.714  rec=0.286  n=35
       YES: f1=0.620  prec=0.595  rec=0.647  n=34


In [10]:

# ==================================================================
# SAVE RESULTS
# ==================================================================
payload = {
    "meta": {
        "version": "v20_1_pure_qwen_v5",
        "model": "Qwen3-8B + LoRA v20",
        "answer_path": "pure_qwen_only",
        "no_z3_no_hybrid_no_lgbm": True,
        "seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "qwen_bon_n": QWEN_BON_N,
        "temperature": QWEN_BON_TEMP,
        "top_p": QWEN_TOP_P,
        "cot_lora_path": COT_LORA_PATH,
        "dataset_path": DATASET_PATH,
    },
    "metrics": {"train": m_train, "val": m_val, "full": m_full},
    "val_predictions": val_pred,
    "train_predictions_sample": train_pred[:20],
}
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_JSON)

# Decision gate
val = m_val
yes_f1 = f1(val, "YES")
no_f1 = f1(val, "NO")
unk_f1 = f1(val, "UNKNOWN")
macro = val["macro_f1"]
print("\n" + "="*70)
print("V20 DECISION GATE")
print("="*70)
print(f"macro-F1   = {macro:.3f}  target >= 0.620")
print(f"No-F1      = {no_f1:.3f}  target >= 0.600")
print(f"Unknown-F1 = {unk_f1:.3f}  target >= 0.550")
print(f"Yes-F1     = {yes_f1:.3f}  target >= 0.750")
if macro >= 0.62 and no_f1 >= 0.60 and unk_f1 >= 0.55 and yes_f1 >= 0.75:
    print("PASS: v20 is a strong final candidate.")
elif macro < 0.538:
    print("FAIL: below v18 macro-F1. Fallback: v20b No 4x, Unknown 1x.")
elif yes_f1 < 0.70:
    print("WARNING: Yes-F1 low. Fallback: reduce No oversample to 2x.")
elif unk_f1 < 0.50:
    print("WARNING: Unknown-F1 low. Fallback: No 3x, Unknown 1.5x.")
else:
    print("MIXED: compare against v17 aggressive and paper defensibility.")
print("="*70)


Saved: /kaggle/working/pipeline_results_20_1_pureqwen_v5.json

V20 DECISION GATE
macro-F1   = 0.641  target >= 0.620
No-F1      = 0.625  target >= 0.600
Unknown-F1 = 0.408  target >= 0.550
Yes-F1     = 0.620  target >= 0.750


In [11]:

# ==================================================================
# OPTIONAL: PREDICT OFFICIAL-LIKE TEST IF PRESENT
# ==================================================================
if os.path.exists(TEST_PATH):
    try:
        test_raw = load_json(TEST_PATH)
        test_rows = iter_questions(test_raw, "test", keep_gold=False)
        print(f"Loaded test: {len(test_raw)} records / {len(test_rows)} questions")
        test_pred = batch_predict(test_rows)
        # CSV: id,answer. Use sample_id_qidx as stable id unless official format differs.
        with open(SUBMISSION_CSV, "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f)
            w.writerow(["id", "answer"])
            for r in test_pred:
                w.writerow([f"{r['sample_id']}_{r['q_idx']}", r["qwen_answer"]])
        with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
            json.dump(test_pred, f, ensure_ascii=False, indent=2)
        print("Saved:", SUBMISSION_CSV)
        print("Saved:", SUBMISSION_JSON)
    except Exception as e:
        print("Test prediction skipped due to error:", repr(e))
else:
    print("TEST_PATH not found, skip official-like prediction.")


Loaded test: 300 records / 600 questions


Rendering prompts:   0%|          | 0/128 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 640/640 [01:43<00:00,  6.19it/s, est. speed input: 1536.58 toks/s, output: 333.44 toks/s]

[128/600] elapsed=1.7 min


Rendering prompts:   0%|          | 0/128 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 640/640 [01:34<00:00,  6.79it/s, est. speed input: 1667.05 toks/s, output: 377.00 toks/s]

[512/600] elapsed=6.6 min


Rendering prompts:   0%|          | 0/88 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 440/440 [01:06<00:00,  6.57it/s, est. speed input: 1597.37 toks/s, output: 365.08 toks/s]

[600/600] elapsed=7.7 min
Saved: /kaggle/working/submission_v20_pureqwen.csv
Saved: /kaggle/working/submission_v20_pureqwen.json
